In [15]:
import os
import glob
import torch
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityd, 
    Resized, ToTensord, RandFlipd, RandRotate90d, RandZoomd
)
from torch.optim.lr_scheduler import CosineAnnealingLR
from monai.data import Dataset, DataLoader
from monai.networks.nets import SwinUNETR
from monai.losses import DiceFocalLoss
from torch.optim import AdamW
from tqdm import tqdm

In [16]:
# 1. Setup Paths and Devices
data_dir = "/home/jiakuny1/Projects/nnUNet_data/nnUNet_raw/Dataset101_Dental"
images_dir = os.path.join(data_dir, "imagesTr")
labels_dir = os.path.join(data_dir, "labelsTr")

# Lock to GPU 1
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

Training on device: cuda:1


In [17]:
# 2. Match Images to Labels
# nnU-Net format: images have _0000.png, labels just have .png
train_images = sorted(glob.glob(os.path.join(images_dir, "*_0000.png")))
train_labels = [img.replace("imagesTr", "labelsTr").replace("_0000.png", ".png") for img in train_images]

data_dicts = [{"image": img, "label": lbl} for img, lbl in zip(train_images, train_labels)]
print(f"Found {len(data_dicts)} training pairs.")

Found 436 training pairs.


In [18]:
# 3. Data Augmentation and Transforms
# Swin UNETR requires spatial dimensions to be divisible by 32. 512x512 is great for X-rays.
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    ScaleIntensityd(keys=["image"]),
    
    # NEW: Aggressive Augmentations to multiply the dataset
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0), # Up/Down flip
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1), # Left/Right flip
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),    # 90-degree rotations
    # Zoom in/out slightly. Image uses bilinear, label uses nearest to preserve class IDs!
    RandZoomd(keys=["image", "label"], prob=0.3, min_zoom=0.8, max_zoom=1.2, mode=("bilinear", "nearest")),
    
    Resized(keys=["image", "label"], spatial_size=(512, 512), mode=("bilinear", "nearest")),
    ToTensord(keys=["image", "label"])
])

train_ds = Dataset(data=data_dicts, transform=train_transforms)
# Using a batch size of 2 or 4 depending on your GPU RAM. Transformers are memory-heavy!
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=4)

In [19]:
# 4. Initialize Swin UNETR
# spatial_dims=2 makes it a 2D network instead of 3D.
model = SwinUNETR(
    in_channels=1,
    out_channels=11, # Background + 10 classes
    feature_size=24, # Capacity of the transformer (can increase to 48 if you have huge VRAM)
    spatial_dims=2
).to(device)

In [20]:
# 5. Loss and Optimizer
# include_background=False is the silver bullet. It forces the model 
# to ONLY care about the teeth and fillings.
loss_function = DiceFocalLoss(
    include_background=False, 
    to_onehot_y=True, 
    softmax=True, 
    squared_pred=True
)

# Transformers are sensitive. We are dropping the learning rate slightly
# to prevent it from bouncing out of the optimal path.
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# NEW: Define how long we are training BEFORE calling the scheduler
max_epochs = 100

# NEW: The Learning Rate Scheduler
scheduler = CosineAnnealingLR(optimizer, T_max=max_epochs)


In [21]:
# 6. The Training Loop
max_epochs = 100 # Start with 100 to see how it learns compared to nnU-Net
model.train()

print("Starting Swin UNETR Training...")
for epoch in range(max_epochs):
    epoch_loss = 0
    step = 0
    
    # Progress bar for the terminal
    epoch_iterator = tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs}", dynamic_ncols=True)
    
    for batch_data in epoch_iterator:
        step += 1
        inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_iterator.set_postfix(loss=loss.item())
        
    epoch_loss /= step
    print(f"Epoch {epoch+1} Average Loss: {epoch_loss:.4f}")

    # NEW: Tell the scheduler to lower the learning rate for the next epoch
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr:.6f}")

    # Save a checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save(model.state_dict(), f"/home/jiakuny1/Projects/swin_unetr_epoch_{epoch+1}.pth")
        print(f"Saved Checkpoint for Epoch {epoch+1}")

print("Training Complete!")

Starting Swin UNETR Training...


Epoch 1/100: 100%|██████████| 218/218 [00:20<00:00, 10.65it/s, loss=1.01] 


Epoch 1 Average Loss: 1.0330
Current Learning Rate: 0.000100


Epoch 2/100: 100%|██████████| 218/218 [00:20<00:00, 10.40it/s, loss=0.946]


Epoch 2 Average Loss: 0.9749
Current Learning Rate: 0.000100


...

Epoch 98 Average Loss: 0.6473
Current Learning Rate: 0.000000


Epoch 99/100: 100%|██████████| 218/218 [00:18<00:00, 12.03it/s, loss=0.592]


Epoch 99 Average Loss: 0.6465
Current Learning Rate: 0.000000


Epoch 100/100: 100%|██████████| 218/218 [00:17<00:00, 12.38it/s, loss=0.506]


Epoch 100 Average Loss: 0.6443
Current Learning Rate: 0.000000
Saved Checkpoint for Epoch 100
Training Complete!
